In [1]:
# ============================================================
# NOTEBOOK 4: RAG RETRIEVAL DEMO
# Project: AgriGenius
#
# Purpose:
# Load the existing ChromaDB vector store and perform semantic retrieval.
#
# Input:
#   outputs/run_<RUN_ID>/vectordb/chroma_db/
#
# Output:
#   outputs/run_<RUN_ID>/rag_demo/search_results.csv
#   outputs/run_<RUN_ID>/rag_demo/search_results.json
#   outputs/run_<RUN_ID>/rag_demo/query_log.json
#   outputs/run_<RUN_ID>/rag_demo/cited_answer_draft.txt
#
# Notes:
# - This notebook does NOT rebuild the vector database.
# - It copies the ChromaDB folder from Drive to /content for safe querying.
# ============================================================


# -----------------------------
# Cell 1 — Install dependencies
# -----------------------------
!pip -q install chromadb==0.5.5 sentence-transformers==3.0.1 pandas pyarrow


# -----------------------------
# Cell 2 — Imports
# -----------------------------
import os
import json
import shutil
from datetime import datetime

import pandas as pd
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings



/usr/local/lib/python3.12/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [2]:
# -----------------------------
# Cell 3 — Mount Google Drive
# -----------------------------
from google.colab import drive
drive.mount("/content/drive", force_remount=True)



Mounted at /content/drive


In [3]:

# -----------------------------
# Cell 4 — Project paths
# -----------------------------
PROJECT_ROOT = "/content/drive/MyDrive/Colab Notebooks/Shruti/AgriGenius_Project"

# ✅ Must be same RUN_ID used in previous notebooks
RUN_ID = "20260209_185402"

RUN_DIR = os.path.join(PROJECT_ROOT, "outputs", f"run_{RUN_ID}")

VECTOR_DIR = os.path.join(RUN_DIR, "vectordb")
CHROMA_DIR_DRIVE = os.path.join(VECTOR_DIR, "chroma_db")

# Copy DB locally before querying
CHROMA_DIR_LOCAL = f"/content/chroma_db_query_{RUN_ID}"

RAG_OUT = os.path.join(RUN_DIR, "rag_demo")
os.makedirs(RAG_OUT, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RUN_DIR:", RUN_DIR)
print("CHROMA_DIR_DRIVE:", CHROMA_DIR_DRIVE, "| exists:", os.path.exists(CHROMA_DIR_DRIVE))
print("CHROMA_DIR_LOCAL:", CHROMA_DIR_LOCAL)
print("RAG_OUT:", RAG_OUT)


# -----------------------------
# Cell 5 — Validate vector DB and copy locally
# -----------------------------
if not os.path.exists(CHROMA_DIR_DRIVE):
    raise FileNotFoundError(
        f"Vector DB not found at: {CHROMA_DIR_DRIVE}\n"
        f"Run Notebook 3 first."
    )

if os.path.exists(CHROMA_DIR_LOCAL):
    shutil.rmtree(CHROMA_DIR_LOCAL)

shutil.copytree(CHROMA_DIR_DRIVE, CHROMA_DIR_LOCAL)

print("✅ Copied vector DB from Drive to local runtime for safe querying.")



PROJECT_ROOT: /content/drive/MyDrive/Colab Notebooks/Shruti/AgriGenius_Project
RUN_DIR: /content/drive/MyDrive/Colab Notebooks/Shruti/AgriGenius_Project/outputs/run_20260209_185402
CHROMA_DIR_DRIVE: /content/drive/MyDrive/Colab Notebooks/Shruti/AgriGenius_Project/outputs/run_20260209_185402/vectordb/chroma_db | exists: True
CHROMA_DIR_LOCAL: /content/chroma_db_query_20260209_185402
RAG_OUT: /content/drive/MyDrive/Colab Notebooks/Shruti/AgriGenius_Project/outputs/run_20260209_185402/rag_demo
✅ Copied vector DB from Drive to local runtime for safe querying.


In [4]:

# -----------------------------
# Cell 6 — Load embedding model
# -----------------------------
EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

embedder = SentenceTransformer(EMBED_MODEL_NAME)
print("✅ Embedding model loaded:", EMBED_MODEL_NAME)


# -----------------------------
# Cell 7 — Connect to local Chroma DB
# -----------------------------
client = chromadb.PersistentClient(
    path=CHROMA_DIR_LOCAL,
    settings=Settings(anonymized_telemetry=False)
)

COLLECTION_NAME = f"agrigenius_{RUN_ID}"

try:
    collection = client.get_collection(COLLECTION_NAME)
except Exception as e:
    raise ValueError(
        f"Collection '{COLLECTION_NAME}' not found inside local Chroma DB.\n"
        f"Please confirm Notebook 3 completed successfully."
    ) from e

print("✅ Chroma collection loaded:", COLLECTION_NAME)
print("Total vectors available:", collection.count())


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


✅ Embedding model loaded: sentence-transformers/all-MiniLM-L6-v2
✅ Chroma collection loaded: agrigenius_20260209_185402
Total vectors available: 891


In [5]:


# -----------------------------
# Cell 8 — Search function
# -----------------------------
def search_vector_db(query: str, k: int = 5):
    """
    Perform semantic similarity search over the Chroma vector store.
    Returns:
        results_df : human-readable DataFrame
        raw_result : raw Chroma response
    """
    query_embedding = embedder.encode([query], show_progress_bar=False).tolist()

    raw_result = collection.query(
        query_embeddings=query_embedding,
        n_results=k,
        include=["documents", "metadatas", "distances"]
    )

    rows = []
    for i in range(len(raw_result["ids"][0])):
        chunk_id = raw_result["ids"][0][i]
        distance = raw_result["distances"][0][i]
        metadata = raw_result["metadatas"][0][i]
        document = raw_result["documents"][0][i]

        rows.append({
            "rank": i + 1,
            "chunk_id": chunk_id,
            "distance": float(distance) if distance is not None else None,
            "source_type": metadata.get("source_type"),
            "source_name": metadata.get("source_name"),
            "file_name": metadata.get("file_name"),
            "chunk_index_in_file": metadata.get("chunk_index_in_file"),
            "chunk_words": metadata.get("chunk_words"),
            "chunk_chars": metadata.get("chunk_chars"),
            "preview": document[:250].replace("\n", " ") + "..."
        })

    results_df = pd.DataFrame(rows)
    return results_df, raw_result



In [6]:

# -----------------------------
# Cell 9 — Demo query
# -----------------------------
QUERY = "What is Minimum Support Price scheme and how does it support farmers?"
TOP_K = 5

print("\n" + "=" * 90)
print("DEMO: SEMANTIC RETRIEVAL")
print("=" * 90)
print("Query:", QUERY)
print("Top-K:", TOP_K)

results_df, raw_result = search_vector_db(QUERY, k=TOP_K)
display(results_df)


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given



DEMO: SEMANTIC RETRIEVAL
Query: What is Minimum Support Price scheme and how does it support farmers?
Top-K: 5


,rank,chunk_id,distance,source_type,source_name,file_name,chunk_index_in_file,chunk_words,chunk_chars,preview
0,1,pdf_Farming_Schemes_0009_000009,0.264845,pdf,Farming_Schemes,Farming_Schemes_pdf.txt,9,220,1340,support prices are a guarantee price for their...
1,2,pdf_Farming_Schemes_0010_000010,0.282286,pdf,Farming_Schemes,Farming_Schemes_pdf.txt,10,220,1527,bumper production years. The minimum support p...
2,3,pdf_Farming_Schemes_0008_000008,0.374984,pdf,Farming_Schemes,Farming_Schemes_pdf.txt,8,220,1307,Support Price Scheme Type of scheme : Central ...
3,4,pdf_Farming_Schemes_0044_000044,0.435858,pdf,Farming_Schemes,Farming_Schemes_pdf.txt,44,220,1504,"be free from diseases, and appear healthy. In ..."
4,5,pdf_Farming_Schemes_0060_000060,0.436235,pdf,Farming_Schemes,Farming_Schemes_pdf.txt,60,220,1503,country. These issues will be raised through v...


In [7]:


# -----------------------------
# Cell 10 — Save retrieval outputs
# -----------------------------
results_csv_path = os.path.join(RAG_OUT, "search_results.csv")
results_json_path = os.path.join(RAG_OUT, "search_results.json")
query_log_path = os.path.join(RAG_OUT, "query_log.json")

results_df.to_csv(results_csv_path, index=False)

with open(results_json_path, "w", encoding="utf-8") as f:
    json.dump(raw_result, f, indent=2)

query_log = {
    "run_id": RUN_ID,
    "created_at": datetime.now().isoformat(),
    "project": "AgriGenius",
    "embedding_model": EMBED_MODEL_NAME,
    "collection_name": COLLECTION_NAME,
    "query": QUERY,
    "top_k": TOP_K,
    "vector_db_path_drive": CHROMA_DIR_DRIVE,
    "vector_db_path_local": CHROMA_DIR_LOCAL,
    "saved_csv": results_csv_path,
    "saved_json": results_json_path
}

with open(query_log_path, "w", encoding="utf-8") as f:
    json.dump(query_log, f, indent=2)

print("✅ Search results saved:", results_csv_path)
print("✅ Raw JSON saved:", results_json_path)
print("✅ Query log saved:", query_log_path)
print(json.dumps(query_log, indent=2))



✅ Search results saved: /content/drive/MyDrive/Colab Notebooks/Shruti/AgriGenius_Project/outputs/run_20260209_185402/rag_demo/search_results.csv
✅ Raw JSON saved: /content/drive/MyDrive/Colab Notebooks/Shruti/AgriGenius_Project/outputs/run_20260209_185402/rag_demo/search_results.json
✅ Query log saved: /content/drive/MyDrive/Colab Notebooks/Shruti/AgriGenius_Project/outputs/run_20260209_185402/rag_demo/query_log.json
{
  "run_id": "20260209_185402",
  "created_at": "2026-03-09T09:12:19.059929",
  "project": "AgriGenius",
  "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
  "collection_name": "agrigenius_20260209_185402",
  "query": "What is Minimum Support Price scheme and how does it support farmers?",
  "top_k": 5,
  "vector_db_path_drive": "/content/drive/MyDrive/Colab Notebooks/Shruti/AgriGenius_Project/outputs/run_20260209_185402/vectordb/chroma_db",
  "vector_db_path_local": "/content/chroma_db_query_20260209_185402",
  "saved_csv": "/content/drive/MyDrive/Colab Noteb

In [8]:

# -----------------------------
# Cell 11 — Build cited answer draft
# -----------------------------
def build_cited_answer(query: str, raw_result: dict, max_sources: int = 3):
    """
    Creates a presentation-ready evidence draft using the top retrieved chunks.
    This is not LLM generation. It is a grounded evidence summary.
    """
    ids = raw_result.get("ids", [[]])[0]
    docs = raw_result.get("documents", [[]])[0]
    metas = raw_result.get("metadatas", [[]])[0]

    lines = []
    lines.append("AgriGenius - Retrieved Evidence Draft")
    lines.append("")
    lines.append(f"User Query: {query}")
    lines.append("")
    lines.append("Top Retrieved Evidence:")
    lines.append("")

    limit = min(max_sources, len(ids))
    for i in range(limit):
        meta = metas[i] if i < len(metas) else {}
        doc = docs[i] if i < len(docs) else ""

        citation = (
            f"[{i+1}] "
            f"file_name={meta.get('file_name')} | "
            f"source_name={meta.get('source_name')} | "
            f"source_type={meta.get('source_type')} | "
            f"chunk_index={meta.get('chunk_index_in_file')} | "
            f"chunk_id={ids[i]}"
        )

        snippet = doc[:700].replace("\n", " ").strip()

        lines.append(citation)
        lines.append(snippet)
        lines.append("")

    lines.append("Note: This draft is generated directly from retrieved evidence chunks for traceable, grounded response construction.")

    return "\n".join(lines).strip()


cited_answer = build_cited_answer(QUERY, raw_result, max_sources=3)

print("\n" + "=" * 90)
print("CITED ANSWER DRAFT")
print("=" * 90)
print(cited_answer)




CITED ANSWER DRAFT
AgriGenius - Retrieved Evidence Draft

User Query: What is Minimum Support Price scheme and how does it support farmers?

Top Retrieved Evidence:

[1] file_name=Farming_Schemes_pdf.txt | source_name=Farming_Schemes | source_type=pdf | chunk_index=9 | chunk_id=pdf_Farming_Schemes_0009_000009
support prices are a guarantee price for their produce from the Government. The major objectives are to support the farmers from distress sales and to procure food grains for public distribution. In case the market price for the commodity falls below the announced minimum price due to bumper production and glut in the market, government agencies purchase the entire quantity offered by the farmers at the announced minimum price. Salient features : If there is a fall in the prices of the crops, after a bumper harvest, the government purchases at the MSP and this is the reason that the priced cannot go below MSP. So this directly helps the farmers. The government decided the support

In [9]:

# -----------------------------
# Cell 12 — Save cited answer draft
# -----------------------------
answer_path = os.path.join(RAG_OUT, "cited_answer_draft.txt")

with open(answer_path, "w", encoding="utf-8") as f:
    f.write(cited_answer)

print("✅ Cited answer draft saved:", answer_path)



✅ Cited answer draft saved: /content/drive/MyDrive/Colab Notebooks/Shruti/AgriGenius_Project/outputs/run_20260209_185402/rag_demo/cited_answer_draft.txt


In [10]:

# -----------------------------
# Cell 13 — Optional multiple-query evaluation
# -----------------------------
EVAL_QUERIES = [
    "What are irrigation schemes for farmers?",
    "What is Minimum Support Price scheme?",
    "What agricultural support programs are available for farmers?",
    "What statistics are available on crop production?",
    "What does the agriculture handbook say about irrigation?"
]

all_eval_rows = []

for query in EVAL_QUERIES:
    eval_df, _ = search_vector_db(query, k=3)
    eval_df.insert(0, "query", query)
    all_eval_rows.append(eval_df)

evaluation_df = pd.concat(all_eval_rows, ignore_index=True)

evaluation_csv_path = os.path.join(RAG_OUT, "evaluation_queries_results.csv")
evaluation_df.to_csv(evaluation_csv_path, index=False)

print("✅ Multi-query evaluation results saved:", evaluation_csv_path)
display(evaluation_df.head(15))

✅ Multi-query evaluation results saved: /content/drive/MyDrive/Colab Notebooks/Shruti/AgriGenius_Project/outputs/run_20260209_185402/rag_demo/evaluation_queries_results.csv


,query,rank,chunk_id,distance,source_type,source_name,file_name,chunk_index_in_file,chunk_words,chunk_chars,preview
0,What are irrigation schemes for farmers?,1,pdf_Farming_Schemes_0054_000054,0.313909,pdf,Farming_Schemes,Farming_Schemes_pdf.txt,54,220,1541,organisations/NGOs; and f) Farmer oriented act...
1,What are irrigation schemes for farmers?,2,pdf_Farming_Schemes_0053_000053,0.338012,pdf,Farming_Schemes,Farming_Schemes_pdf.txt,53,220,1630,strategize by focussing on end -to end solutio...
2,What are irrigation schemes for farmers?,3,pdf_farmerbook_0046_000605,0.344294,pdf,farmerbook,farmerbook_pdf.txt,46,220,1363,(40% by the Government of India and 10% by the...
3,What is Minimum Support Price scheme?,1,pdf_Farming_Schemes_0009_000009,0.387263,pdf,Farming_Schemes,Farming_Schemes_pdf.txt,9,220,1340,support prices are a guarantee price for their...
4,What is Minimum Support Price scheme?,2,pdf_Farming_Schemes_0010_000010,0.400016,pdf,Farming_Schemes,Farming_Schemes_pdf.txt,10,220,1527,bumper production years. The minimum support p...
5,What is Minimum Support Price scheme?,3,pdf_Farming_Schemes_0008_000008,0.459804,pdf,Farming_Schemes,Farming_Schemes_pdf.txt,8,220,1307,Support Price Scheme Type of scheme : Central ...
6,What agricultural support programs are availab...,1,pdf_Farming_Schemes_0044_000044,0.371046,pdf,Farming_Schemes,Farming_Schemes_pdf.txt,44,220,1504,"be free from diseases, and appear healthy. In ..."
7,What agricultural support programs are availab...,2,pdf_farmerbook_0257_000816,0.397179,pdf,farmerbook,farmerbook_pdf.txt,257,220,1669,"Universities, ICAR Institutions, Commodity Boa..."
8,What agricultural support programs are availab...,3,pdf_farmerbook_0242_000801,0.398077,pdf,farmerbook,farmerbook_pdf.txt,242,220,1500,beings and livestock. 4. Dispose the pesticide...
9,What statistics are available on crop production?,1,web_Production_of_Horticultural_Crops_0001_000887,0.338255,web,Production_of_Horticultural_Crops,web_Production_of_Horticultural_Crops.txt,1,220,1463,SECTION: Production of Horticultural Crops URL...
